# ANN Book Genre Classification

Run all cells from top to bottom to load the dataset, split it correctly, train the ANN, save a checkpoint, and print the final metrics.

## 1. Load Dataset

In [ ]:
from pathlib import Path
import json
import joblib
import pandas as pd

from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

candidate_paths = [
    Path.cwd() / "BooksClassifier_dataset_cleaned.csv.xls",
    Path.cwd().parent / "BooksClassifier_dataset_cleaned.csv.xls",
]

DATA_PATH = next((path for path in candidate_paths if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Cleaned dataset not found. Keep the ANN folder inside the main project folder.")

raw_df = pd.read_csv(DATA_PATH)
df = raw_df[["title", "cleaned_text", "target_genre"]].dropna().copy()
df["title"] = df["title"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df["cleaned_text"] = df["cleaned_text"].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
df = df[(df["title"] != "") & (df["cleaned_text"] != "")].copy()
duplicates_removed = int(df.duplicated(subset=["cleaned_text"]).sum())
df = df.drop_duplicates(subset=["cleaned_text"], keep="first").reset_index(drop=True)
df["model_text"] = df["title"] + " " + df["cleaned_text"]

print("Dataset path:", DATA_PATH)
print("Original rows:", len(raw_df))
print("Rows used:", len(df))
print("Duplicate text rows removed:", duplicates_removed)
print("Genres:", sorted(df["target_genre"].unique()))

display(df[["title", "target_genre", "cleaned_text"]].head())
display(df["target_genre"].value_counts().rename_axis("genre").reset_index(name="rows"))

## 2. Split Dataset

In [ ]:
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df["target_genre"])
X = df["model_text"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

print("Train rows:", len(X_train))
print("Validation rows:", len(X_val))
print("Test rows:", len(X_test))

## 3. Model Cell

In [ ]:
EPOCHS = 70
PATIENCE = 6

ann_model = Pipeline([
    ("features", FeatureUnion([
        ("word_tfidf", TfidfVectorizer(max_features=12000, ngram_range=(1, 2), min_df=2, sublinear_tf=True)),
        ("char_tfidf", TfidfVectorizer(analyzer="char_wb", max_features=4000, ngram_range=(3, 5), min_df=2, sublinear_tf=True)),
    ])),
    ("classifier", MLPClassifier(
        hidden_layer_sizes=(128,),
        activation="relu",
        solver="adam",
        alpha=0.0005,
        batch_size=32,
        learning_rate_init=0.001,
        max_iter=EPOCHS,
        early_stopping=True,
        validation_fraction=0.15,
        n_iter_no_change=PATIENCE,
        random_state=RANDOM_STATE,
        verbose=True,
    )),
])

ann_model

## 4. Training Cell With Epochs, Early Stopping, And Checkpoint

In [ ]:
print("Training ANN")
print("Maximum epochs:", EPOCHS)
print("Early stopping patience:", PATIENCE)

ann_model.fit(X_train, y_train)
joblib.dump(ann_model, "genre_ann_checkpoint.joblib")

classifier = ann_model.named_steps["classifier"]
print("Epochs completed:", classifier.n_iter_)
print("Checkpoint saved: genre_ann_checkpoint.joblib")

## 5. Results: Accuracy, Macro F1, Weighted F1, And Classification Report

In [ ]:
val_predictions = ann_model.predict(X_val)
test_predictions = ann_model.predict(X_test)

val_accuracy = accuracy_score(y_val, val_predictions)
test_accuracy = accuracy_score(y_test, test_predictions)
test_macro_f1 = f1_score(y_test, test_predictions, average="macro", zero_division=0)
test_weighted_f1 = f1_score(y_test, test_predictions, average="weighted", zero_division=0)

metrics = {
    "model": "ANN",
    "validation_accuracy": float(val_accuracy),
    "test_accuracy": float(test_accuracy),
    "test_macro_f1": float(test_macro_f1),
    "test_weighted_f1": float(test_weighted_f1),
    "epochs_completed": int(ann_model.named_steps["classifier"].n_iter_),
}

print("Validation accuracy:", round(val_accuracy, 4))
print("Test accuracy:", round(test_accuracy, 4))
print("Test macro F1:", round(test_macro_f1, 4))
print("Test weighted F1:", round(test_weighted_f1, 4))
print()
print(classification_report(y_test, test_predictions, target_names=label_encoder.classes_, zero_division=0))

with open("ann_metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)